# TikTok Data Pipeline
Flow: Manifest → Audio → Transcripts → One Analysis CSV

This notebook pulls **public per-video counts** from my TikTok profile (via `yt-dlp`), downloads audio, transcribes with Whisper, and writes **one final CSV** that includes:
- `video_id` (primary key)
- views/likes/comments/reposts (when available publicly)
- timestamps/dates
- `transcript` (single string column)

**Caching behavior (important):**
- If an audio file already exists for a video, we **skip re-downloading** it.
- If a transcript already exists for a video, we **skip re-transcribing** it.

That means you can re-run this notebook regularly and it will only process new videos.


## 1. Setup and configuration
- Set your handle and key paths
- Create `data/` subfolders if they don’t exist
- (Optional) tweak the Whisper model size


In [1]:
import json, subprocess, warnings
from pathlib import Path

import pandas as pd

# --------------------
# CONFIG
# --------------------
HANDLE = "humbletoker"
PROFILE_URL = f"https://www.tiktok.com/@{HANDLE}"

# Project folders (adjust if your repo structure differs)
DATA_DIR = Path("../data")
AUDIO_DIR = DATA_DIR / "audio"
DERIVED_DIR = DATA_DIR / "derived"

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

# Outputs
MANIFEST_CSV = DERIVED_DIR / "tiktok_manifest.csv"
TRANSCRIPTS_CSV = DERIVED_DIR / "tiktok_transcripts.csv"  # cache
FINAL_CSV = DERIVED_DIR / "tiktok_manifest_with_transcripts.csv"

print("PROFILE_URL:", PROFILE_URL)
print("AUDIO_DIR:", AUDIO_DIR.resolve())
print("DERIVED_DIR:", DERIVED_DIR.resolve())


PROFILE_URL: https://www.tiktok.com/@humbletoker
AUDIO_DIR: /Users/Logan/Desktop/TikTok Goodnight/tiktok-analysis/data/audio
DERIVED_DIR: /Users/Logan/Desktop/TikTok Goodnight/tiktok-analysis/data/derived


## 2. Scrape video manifest (public metadata)

We use `yt-dlp --dump-json` on your profile to collect per-video metadata.

Notes:
- TikTok does **not** reliably expose `share_count` publicly, so we don’t include it here.
- `view_count / like_count / comment_count / repost_count` are often available and are enough to compute engagement rates.


In [2]:
cmd = ["yt-dlp", "--dump-json", PROFILE_URL]
proc = subprocess.run(cmd, capture_output=True, text=True)

if proc.returncode != 0:
    raise RuntimeError(proc.stderr[:4000])

rows = []
for line in proc.stdout.splitlines():
    try:
        obj = json.loads(line)
    except json.JSONDecodeError:
        continue

    url = obj.get("webpage_url") or obj.get("original_url") or ""
    vid = obj.get("id")

    if not vid and url:
        m = pd.Series([url]).astype(str).str.extract(r"/video/(\d+)")
        vid = m.iloc[0, 0]

    if not vid:
        continue

    rows.append({
        "video_id": str(vid),
        "video_url": url,
        "title": obj.get("title") or "",
        "description": obj.get("description") or "",
        "upload_date": obj.get("upload_date"),
        "timestamp": obj.get("timestamp"),
        "duration": obj.get("duration"),
        "view_count": obj.get("view_count"),
        "like_count": obj.get("like_count"),
        "comment_count": obj.get("comment_count"),
        "repost_count": obj.get("repost_count"),
    })

videos = pd.DataFrame(rows).drop_duplicates(subset=["video_id"]).reset_index(drop=True)

# Type cleanup
videos["video_id"] = videos["video_id"].astype("string")
for c in ["view_count", "like_count", "comment_count", "repost_count", "duration", "timestamp", "upload_date"]:
    if c in videos.columns:
        videos[c] = pd.to_numeric(videos[c], errors="coerce")

# Helpful datetime columns
if "timestamp" in videos.columns:
    videos["posted_at_utc"] = pd.to_datetime(videos["timestamp"], unit="s", utc=True, errors="coerce")
    videos["posted_at_local"] = videos["posted_at_utc"].dt.tz_convert("America/Detroit")

if "upload_date" in videos.columns:
    videos["upload_date_dt"] = pd.to_datetime(
        videos["upload_date"].astype("Int64").astype("string"),
        format="%Y%m%d",
        errors="coerce",
    )

videos.to_csv(MANIFEST_CSV, index=False)
print(f"Saved manifest: {MANIFEST_CSV.resolve()}")
print("Videos found:", len(videos))
videos.head()


Saved manifest: /Users/Logan/Desktop/TikTok Goodnight/tiktok-analysis/data/derived/tiktok_manifest.csv
Videos found: 53


,video_id,video_url,title,description,upload_date,timestamp,duration,view_count,like_count,comment_count,repost_count,posted_at_utc,posted_at_local,upload_date_dt
0,7594277449698004238,https://www.tiktok.com/@humbletoker/video/7594...,"Goodnight to everyone EXCEPT challange, level ...","Goodnight to everyone EXCEPT challange, level ...",20260112,1768180540,64,656,28,3,0,2026-01-12 01:15:40+00:00,2026-01-11 20:15:40-05:00,2026-01-12
1,7593500470837087501,https://www.tiktok.com/@humbletoker/video/7593...,TikTok video #7593500470837087501,,20260109,1767999638,73,320,18,3,0,2026-01-09 23:00:38+00:00,2026-01-09 18:00:38-05:00,2026-01-09
2,7592046457117625613,https://www.tiktok.com/@humbletoker/video/7592...,This may be easier than 2% actually… goodnight...,This may be easier than 2% actually… goodnight...,20260106,1767661102,63,1339,53,9,63,2026-01-06 00:58:22+00:00,2026-01-05 19:58:22-05:00,2026-01-06
3,7542733065643232526,https://www.tiktok.com/@humbletoker/video/7542...,Yeezy SpongeBob hamburger XQC. Comment if you ...,Yeezy SpongeBob hamburger XQC. Comment if you ...,20250826,1756179438,62,1965,68,11,72,2025-08-26 03:37:18+00:00,2025-08-25 23:37:18-04:00,2025-08-26
4,7542350925122293006,https://www.tiktok.com/@humbletoker/video/7542...,"Goodnight to everyone except… Labubu, Africa, ...","Goodnight to everyone except… Labubu, Africa, ...",20250825,1756090455,62,8285,137,31,343,2025-08-25 02:54:15+00:00,2025-08-24 22:54:15-04:00,2025-08-25


## 3. Download audio (skip if already present)

For each video, we download audio-only to `{DATA_DIR}/audio/<video_id>.m4a`.

If `<video_id>.m4a` already exists and is non-empty, we skip it.


In [3]:
failed_ids = []

cache_hits = 0
for _, r in videos.iterrows():
    vid = str(r["video_id"])
    url = r["video_url"]
    out_m4a = AUDIO_DIR / f"{vid}.m4a"

    if out_m4a.exists() and out_m4a.stat().st_size > 0:
        cache_hits += 1
        continue

    cmd = [
        "yt-dlp", url,
        "--no-playlist",
        "-x", "--audio-format", "m4a", "--audio-quality", "0",
        "-o", str(AUDIO_DIR / f"{vid}.%(ext)s"),
        "--force-overwrites",
        "--no-warnings",
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)

    if proc.returncode != 0 or (not out_m4a.exists()) or out_m4a.stat().st_size == 0:
        failed_ids.append(vid)

print("Audio cache hits:", cache_hits)
print("Audio download failed:", len(failed_ids))
failed_ids[:10]


Audio cache hits: 53
Audio download failed: 0


[]

## 4. Salvage failed downloads with ffmpeg

Some TikToks download as video-only streams or formats that `yt-dlp` can't extract cleanly on the first pass.
For failures, we:
1. download the MP4
2. use `ffmpeg` to extract audio to M4A

If the MP4 contains **no audio stream**, salvage will still fail (that video likely has audio removed).


In [4]:
def ffmpeg_extract_m4a(mp4_path: Path, m4a_path: Path) -> bool:
    # Return True if extracted audio successfully.
    cmd = [
        "ffmpeg", "-y",
        "-i", str(mp4_path),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        str(m4a_path),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    return proc.returncode == 0 and m4a_path.exists() and m4a_path.stat().st_size > 0

still_failed = []

for vid in failed_ids:
    url = f"https://www.tiktok.com/@{HANDLE}/video/{vid}"
    mp4_path = AUDIO_DIR / f"{vid}.mp4"
    m4a_path = AUDIO_DIR / f"{vid}.m4a"

    # Download MP4
    cmd = [
        "yt-dlp", url,
        "--no-playlist",
        "-o", str(mp4_path),
        "--force-overwrites",
        "--no-warnings",
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0 or (not mp4_path.exists()) or mp4_path.stat().st_size == 0:
        still_failed.append(vid)
        continue

    ok = ffmpeg_extract_m4a(mp4_path, m4a_path)

    # Cleanup mp4 to save space
    try:
        mp4_path.unlink(missing_ok=True)
    except Exception:
        pass

    if not ok:
        still_failed.append(vid)

print("Recovered via salvage:", len(failed_ids) - len(still_failed))
print("Still failing:", len(still_failed))
still_failed


Recovered via salvage: 0
Still failing: 0


[]

## 5. Transcribe audio with Whisper (skip if transcript already exists)

We maintain a **transcript cache** at `{DERIVED_DIR}/tiktok_transcripts.csv`.

On each run we:
- load existing transcripts (if any)
- transcribe only missing `video_id`s that have audio files
- write the updated transcripts cache back to disk


In [5]:
import whisper

warnings.filterwarnings(
    "ignore",
    message="FP16 is not supported on CPU; using FP32 instead",
    category=UserWarning,
)

# Load transcript cache (if it exists)
if TRANSCRIPTS_CSV.exists():
    transcripts_cache = pd.read_csv(TRANSCRIPTS_CSV, dtype={"video_id": "string"})
else:
    transcripts_cache = pd.DataFrame(columns=["video_id", "transcript"], dtype="string")

transcripts_cache["video_id"] = transcripts_cache["video_id"].astype("string")
transcripts_cache["transcript"] = transcripts_cache.get("transcript", "").fillna("").astype("string")

cache_map = dict(zip(transcripts_cache["video_id"].astype(str), transcripts_cache["transcript"].astype(str)))

audio_files = sorted(AUDIO_DIR.glob("*.m4a"))
print("Audio files present:", len(audio_files))

to_transcribe = [
    p for p in audio_files
    if (p.stem not in cache_map) or (not str(cache_map.get(p.stem, "")).strip())
]
print("Need transcription:", len(to_transcribe))

model = whisper.load_model("base")  # change to "small" for better accuracy

new_rows = []
for p in to_transcribe:
    vid = p.stem
    try:
        result = model.transcribe(str(p), fp16=False)
        text = (result.get("text") or "").strip()
    except Exception:
        text = ""
    new_rows.append({"video_id": vid, "transcript": text})

if new_rows:
    transcripts_cache = pd.concat([transcripts_cache, pd.DataFrame(new_rows)], ignore_index=True)

# Keep one row per video_id (prefer non-empty transcript)
transcripts_cache["transcript_len"] = transcripts_cache["transcript"].fillna("").str.len()
transcripts_cache = (transcripts_cache
                     .sort_values(["video_id", "transcript_len"])
                     .drop_duplicates(subset=["video_id"], keep="last")
                     .drop(columns=["transcript_len"])
                     .reset_index(drop=True))

transcripts_cache.to_csv(TRANSCRIPTS_CSV, index=False)
print(f"Saved transcripts cache: {TRANSCRIPTS_CSV.resolve()}")
transcripts_cache.head()


Audio files present: 53
Need transcription: 0
Saved transcripts cache: /Users/Logan/Desktop/TikTok Goodnight/tiktok-analysis/data/derived/tiktok_transcripts.csv


,video_id,transcript
0,7508222226462854446,Good night to everyone except 90% of people wi...
1,7508572618233105707,Good night to everybody except 97% of people w...
2,7509317257672101166,Good night to everyone except 96% of people wi...
3,7509719616160042286,Good night to everyone except 97% of people wi...
4,7510087154660314398,Good night to everyone except 95% of people wi...


## 6. Build one analysis-ready CSV (manifest + transcript)

This is the file you’ll use in your analysis notebook(s).
We merge on `video_id` and keep the transcript as the **last column** for readability.


In [6]:
final = videos.merge(transcripts_cache, on="video_id", how="left")

# Put transcript as last column
if "transcript" in final.columns:
    cols = [c for c in final.columns if c != "transcript"] + ["transcript"]
    final = final[cols]

final.to_csv(FINAL_CSV, index=False)
print(f"Saved final dataset: {FINAL_CSV.resolve()}")
print("Rows:", len(final), "| Columns:", len(final.columns))
final.head()


Saved final dataset: /Users/Logan/Desktop/TikTok Goodnight/tiktok-analysis/data/derived/tiktok_manifest_with_transcripts.csv
Rows: 53 | Columns: 15


,video_id,video_url,title,description,upload_date,timestamp,duration,view_count,like_count,comment_count,repost_count,posted_at_utc,posted_at_local,upload_date_dt,transcript
0,7594277449698004238,https://www.tiktok.com/@humbletoker/video/7594...,"Goodnight to everyone EXCEPT challange, level ...","Goodnight to everyone EXCEPT challange, level ...",20260112,1768180540,64,656,28,3,0,2026-01-12 01:15:40+00:00,2026-01-11 20:15:40-05:00,2026-01-12,Good night to everyone except 99% of people lo...
1,7593500470837087501,https://www.tiktok.com/@humbletoker/video/7593...,TikTok video #7593500470837087501,,20260109,1767999638,73,320,18,3,0,2026-01-09 23:00:38+00:00,2026-01-09 18:00:38-05:00,2026-01-09,Good night to everyone except for the followin...
2,7592046457117625613,https://www.tiktok.com/@humbletoker/video/7592...,This may be easier than 2% actually… goodnight...,This may be easier than 2% actually… goodnight...,20260106,1767661102,63,1339,53,9,63,2026-01-06 00:58:22+00:00,2026-01-05 19:58:22-05:00,2026-01-06,Good night to everyone except for the followin...
3,7542733065643232526,https://www.tiktok.com/@humbletoker/video/7542...,Yeezy SpongeBob hamburger XQC. Comment if you ...,Yeezy SpongeBob hamburger XQC. Comment if you ...,20250826,1756179438,62,1965,68,11,72,2025-08-26 03:37:18+00:00,2025-08-25 23:37:18-04:00,2025-08-26,Today I'm gonna be giving good nights to every...
4,7542350925122293006,https://www.tiktok.com/@humbletoker/video/7542...,"Goodnight to everyone except… Labubu, Africa, ...","Goodnight to everyone except… Labubu, Africa, ...",20250825,1756090455,62,8285,137,31,343,2025-08-25 02:54:15+00:00,2025-08-24 22:54:15-04:00,2025-08-25,Good night to everyone except for the followin...


## 7. Quick sanity check + example transcript


In [7]:
have = final["transcript"].fillna("").str.strip().astype(bool).sum()
print("Videos with transcripts:", have, "/", len(final))

example = final.loc[final["transcript"].fillna("").str.strip() != "", ["video_id", "title", "transcript"]].head(1)
example


Videos with transcripts: 53 / 53


,video_id,title,transcript
0,7594277449698004238,"Goodnight to everyone EXCEPT challange, level ...",Good night to everyone except 99% of people lo...
